# 📊 Proyecto Final - Sistema Inteligente de Recomendación para E-commerce

# Notebook 04: Preparación de Datos (Data Preparation)

---

## 👥 Equipo

- Yustin Rodríguez
- Elías
- Carina
- Rocío

---

## 🎯 Objetivo

Preparar el conjunto de datos para el entrenamiento de los modelos de recomendación mediante procesos de limpieza, integración, transformación y validación de la información.

Esta etapa garantiza que los datos sean consistentes, completos y adecuados para el proceso de modelado.

---

## 📌 Objetivos específicos

- Integrar las tablas necesarias para el modelado.
- Tratar valores nulos e inconsistencias.
- Eliminar registros duplicados.
- Seleccionar las variables relevantes.
- Transformar variables categóricas cuando sea necesario.
- Normalizar o escalar variables (si aplica).
- Construir el dataset final para entrenamiento y validación.

---

## 📂 Datos utilizados

- customers.csv
- sessions.csv
- products.csv
- orders.csv
- order_items.csv
- reviews.csv
- events.csv

---

## 📋 Actividades desarrolladas

- Integración de tablas.
- Limpieza de datos.
- Tratamiento de valores faltantes.
- Eliminación de duplicados.
- Conversión de tipos de datos.
- Validación de consistencia.
- Preparación del dataset final.

---

## 📈 Resultado esperado

Al finalizar este notebook se obtendrá un dataset limpio, integrado y preparado para el entrenamiento de los modelos de recomendación desarrollados en la siguiente fase del proyecto.

---

**Metodología:** CRISP-DM

**Sprint:** 1 – Preparación de Datos

## 1. Carga de datos

Se cargan `order_items`, `orders`, `products` y `customers` desde `Data/Raw/`. El resto de los datasets (`sessions`, `reviews`, `events`) se integran en pasos posteriores.

La carga y la limpieza reutilizan funciones de `SRC/data_loader.py` y `SRC/data_clean.py`, para que la misma lógica se pueda ejecutar fuera del notebook (scripts, pipelines) sin duplicar código.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / 'SRC'))
from data_loader import load_order_items, load_orders, load_products, load_customers
from data_clean import clean_order_items, clean_orders, clean_products, clean_customers, corregir_coherencia_temporal

pd.set_option('display.width', 140)

order_items = load_order_items()
orders = load_orders()
products = load_products()
customers = load_customers()

In [2]:
order_items.head()

,order_id,product_id,unit_price_usd,quantity,line_total_usd
0,1,226,107.15,1,107.15
1,2,771,116.17,1,116.17
2,3,415,94.49,1,94.49
3,3,24,42.86,1,42.86
4,4,1157,32.18,1,32.18


## 2. Limpieza de `order_items`

### 2.1 Diagnóstico inicial

Antes de limpiar, se revisa la estructura, los tipos de dato, los valores nulos y las filas duplicadas. Esto permite decidir qué pasos de limpieza son realmente necesarios en lugar de aplicar transformaciones a ciegas.

In [3]:
print('Shape:', order_items.shape)
print()
print(order_items.dtypes)
print()
print('Nulos por columna:')
print(order_items.isnull().sum())
print()
print('Filas duplicadas (exactas):', order_items.duplicated().sum())

Shape: (59163, 5)

order_id            int64
product_id          int64
unit_price_usd    float64
quantity            int64
line_total_usd    float64
dtype: object

Nulos por columna:
order_id          0
product_id        0
unit_price_usd    0
quantity          0
line_total_usd    0
dtype: int64

Filas duplicadas (exactas): 73


**Resultado:** no hay valores nulos y los tipos de dato ya son correctos (`order_id`/`product_id`/`quantity` como enteros, precios como float), por lo que no hace falta convertir tipos. Sí aparecen 73 filas exactamente duplicadas (mismo `order_id`, `product_id`, precio y cantidad).

### 2.2 Eliminación de duplicados

Una fila duplicada exacta en `order_items` significa que la misma línea de compra (mismo pedido, mismo producto, mismo precio y cantidad) quedó registrada dos veces. Esto no representa una segunda compra real, sino un error de carga/duplicación de datos.

`clean_order_items` (en `SRC/data_clean.py`) elimina esos duplicados y además valida que no haya nulos, valores negativos, inconsistencias de cálculo ni referencias huérfanas — cada validación se imprime en pantalla (cantidad de registros que la incumplen), sin interrumpir la ejecución si algo falla.

In [4]:
before = len(order_items)
order_items = clean_order_items(order_items, products, orders)
after = len(order_items)

print(f'Filas antes: {before}')
print(f'Filas despues: {after}')
print(f'Filas eliminadas: {before - after}')

Filas antes: 59163
Filas despues: 59090
Filas eliminadas: 73


### 2.3 Validación de valores fuera de rango

Se verifica que no existan precios, cantidades o totales negativos o en cero, ya que en un ítem de compra estos valores no tienen sentido de negocio y podrían indicar errores de carga o registros de prueba.

In [5]:
print('unit_price_usd <= 0:', (order_items['unit_price_usd'] <= 0).sum())
print('quantity <= 0:', (order_items['quantity'] <= 0).sum())
print('line_total_usd <= 0:', (order_items['line_total_usd'] <= 0).sum())

unit_price_usd <= 0: 0
quantity <= 0: 0
line_total_usd <= 0: 0


**Resultado:** no se encontraron valores negativos ni en cero en ninguna de las tres columnas, por lo que no se elimina ni corrige ninguna fila en este paso.

### 2.4 Validación de consistencia (`line_total_usd = unit_price_usd × quantity`)

`line_total_usd` es una columna derivada de `unit_price_usd` y `quantity`. Se recalcula y se compara contra el valor almacenado para detectar inconsistencias de cálculo (por ejemplo, precios actualizados después de calcular el total).

In [6]:
calculado = (order_items['unit_price_usd'] * order_items['quantity']).round(2)
diferencia = (calculado - order_items['line_total_usd']).abs()

inconsistentes = order_items[diferencia > 0.01]
print('Filas inconsistentes:', len(inconsistentes))
inconsistentes.head()

Filas inconsistentes: 0


,order_id,product_id,unit_price_usd,quantity,line_total_usd


**Resultado:** no hay filas inconsistentes; `line_total_usd` coincide exactamente con `unit_price_usd × quantity` en el 100% de los casos, así que no se recalcula ni corrige nada.

### 2.5 Validación de integridad referencial

Se comprueba que todo `product_id` exista en `products` y todo `order_id` exista en `orders`. Un `order_items` con referencias inválidas ('huérfanas') no se podría integrar correctamente con el resto de las tablas.

In [7]:
product_huerfanos = (~order_items['product_id'].isin(products['product_id'])).sum()
order_huerfanos = (~order_items['order_id'].isin(orders['order_id'])).sum()

print('product_id sin match en products:', product_huerfanos)
print('order_id sin match en orders:', order_huerfanos)

product_id sin match en products: 0
order_id sin match en orders: 0


**Resultado:** no se encontraron referencias huérfanas; todas las líneas de `order_items` corresponden a un producto y un pedido existentes.

### 2.6 Resumen de la limpieza de `order_items`

- **Nulos:** no había, no se aplicó imputación.
- **Duplicados:** se eliminaron 73 filas duplicadas exactas.
- **Valores fuera de rango:** no se encontraron precios, cantidades ni totales negativos o en cero.
- **Consistencia de cálculo:** `line_total_usd` es 100% consistente con `unit_price_usd × quantity`.
- **Integridad referencial:** todas las filas tienen `product_id` y `order_id` válidos.



In [8]:
orders.head()

,order_id,customer_id,order_time,payment_method,discount_pct,subtotal_usd,total_usd,country,device,source
0,1,13917,2025-01-31T23:07:42,card,20,107.15,85.72,PL,desktop,organic
1,2,1022,2024-02-19T01:17:50,card,0,116.17,116.17,FR,tablet,organic
2,3,6145,2024-12-04T20:24:13,card,0,137.35,137.35,US,mobile,organic
3,4,3152,2024-07-17T08:50:47,card,15,32.18,27.35,BR,mobile,email
4,5,12378,2020-08-21T16:54:16,card,0,238.09,238.09,NL,desktop,paid


## 3. Limpieza de `orders`

### 3.1 Diagnóstico inicial

Se revisan estructura, tipos, nulos y duplicados.

In [9]:
print('Shape:', orders.shape)
print()
print(orders.dtypes)
print()
print('Nulos por columna:')
print(orders.isnull().sum())
print()
print('Filas duplicadas (exactas):', orders.duplicated().sum())
print('order_id duplicados:', orders['order_id'].duplicated().sum())

Shape: (33580, 10)

order_id            int64
customer_id         int64
order_time            str
payment_method        str
discount_pct        int64
subtotal_usd      float64
total_usd         float64
country               str
device                str
source                str
dtype: object

Nulos por columna:
order_id          0
customer_id       0
order_time        0
payment_method    0
discount_pct      0
subtotal_usd      0
total_usd         0
country           0
device            0
source            0
dtype: int64



Filas duplicadas (exactas):

 0
order_id duplicados: 0


**Resultado:** sin nulos, sin filas duplicadas y `order_id` es único (una fila por pedido).

### 3.2 Validación de valores categóricos

Se listan los valores únicos de las columnas categóricas (`payment_method`, `discount_pct`, `country`, `device`, `source`) para detectar variantes inconsistentes (mayúsculas/minúsculas distintas, espacios, categorías inesperadas).

In [10]:
print('payment_method:', sorted(orders['payment_method'].unique()))
print('discount_pct:', sorted(orders['discount_pct'].unique()))
print('device:', sorted(orders['device'].unique()))
print('source:', sorted(orders['source'].unique()))
print('country:', sorted(orders['country'].unique()))

payment_method: ['card', 'cod', 'paypal', 'wallet']
discount_pct: [np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20)]
device: ['desktop', 'mobile', 'tablet']
source: ['direct', 'email', 'organic', 'paid', 'referral', 'social']
country: ['AE', 'AU', 'BR', 'CA', 'DE', 'ES', 'FR', 'GB', 'IN', 'JP', 'MX', 'NL', 'PL', 'SE', 'SG', 'US', 'ZA']


**Resultado:** todas las columnas categóricas tienen valores limpios y acotados (4 métodos de pago, descuentos entre 0% y 20%, 3 dispositivos, 6 canales de origen, 17 países). No hay variantes de formato ni categorías espurias.

### 3.3 Validación de valores numéricos fuera de rango

Se verifica que `subtotal_usd` y `total_usd` no sean negativos ni cero, y que `discount_pct` esté entre 0 y 100.

In [11]:
print('subtotal_usd <= 0:', (orders['subtotal_usd'] <= 0).sum())
print('total_usd <= 0:', (orders['total_usd'] <= 0).sum())
print('discount_pct fuera de [0, 100]:', ((orders['discount_pct'] < 0) | (orders['discount_pct'] > 100)).sum())

subtotal_usd <= 0: 0
total_usd <= 0: 0
discount_pct fuera de [0, 100]: 0


**Resultado:** no se encontraron valores fuera de rango.

### 3.4 Validación de consistencia (`total_usd = subtotal_usd × (1 − discount_pct / 100)`)

Se recalcula el total esperado a partir del subtotal y el descuento, y se compara contra `total_usd` registrado.

In [12]:
calculado = (orders['subtotal_usd'] * (1 - orders['discount_pct'] / 100)).round(2)
diferencia = (calculado - orders['total_usd']).abs()

print('Filas con diferencia > 0.01:', (diferencia > 0.01).sum())
print('Filas con diferencia > 0 (cualquier diferencia):', (diferencia > 0).sum())
print('Diferencia maxima observada:', diferencia.max())

Filas con diferencia > 0.01:

 110
Filas con diferencia > 0 (cualquier diferencia): 303
Diferencia maxima observada: 0.010000000000104592


**Resultado:** hay 110 filas con una diferencia de exactamente $0.01 (un centavo, en más o en menos) entre el total calculado y el registrado. No hay ninguna diferencia mayor a un centavo. Esto es ruido de redondeo de punto flotante (no un error de negocio), así que **no se modifica** `total_usd`: se documenta la validación pero se conserva el valor original registrado.

### 3.5 Validación de fechas

Se confirma que `order_time` sea siempre parseable como fecha y que el rango de fechas sea razonable para el negocio.

In [13]:
order_time_parsed = pd.to_datetime(orders['order_time'], errors='coerce')
print('Fechas no parseables:', order_time_parsed.isnull().sum())
print('Rango de fechas:', order_time_parsed.min(), '->', order_time_parsed.max())

Fechas no parseables:

 0
Rango de fechas: 2020-01-01 01:10:58 -> 2025-10-31 22:59:41


**Resultado:** todas las fechas son válidas y el rango (2020-2025) es coherente con la operación del negocio; no hay fechas futuras ni valores absurdos.

### 3.6 Validación de integridad referencial

Se comprueba que todo `customer_id` en `orders` exista en `customers`.

In [14]:
customer_huerfanos = (~orders['customer_id'].isin(customers['customer_id'])).sum()
print('customer_id sin match en customers:', customer_huerfanos)

customer_id sin match en customers: 0


**Resultado:** no hay `customer_id` huérfanos.

### 3.7 Corrección de coherencia temporal (`order_time` vs `signup_date`)

El EDA de Rocío (`Notebooks/rocio/eda.ipynb`, sección 3.1) detectó que el 50.4% de los pedidos tienen `order_time` anterior al `signup_date` del cliente, y definió el criterio de corrección para todo el equipo: reasignar la fecha inconsistente a `signup_date + offset aleatorio de 0 a 365 días` (semilla fija 42), el mismo método que ella aplica a `review_time`, `start_time` y `timestamp` en su módulo de events/sessions/reviews.

`corregir_coherencia_temporal` (en `SRC/data_clean.py`) aplica ese mismo criterio a `order_time`, que es la tabla que le corresponde a este módulo.

In [ ]:
orders = corregir_coherencia_temporal(orders, customers)

### 3.8 Resumen de la limpieza de `orders`

- **Nulos y duplicados:** no había.
- **Valores categóricos:** consistentes, sin variantes de formato.
- **Valores fuera de rango:** no se encontraron.
- **Consistencia de cálculo:** diferencias de ±$0.01 en 110 filas, atribuibles a redondeo; no ameritan corrección.
- **Fechas:** todas válidas, rango 2020-2025.
- **Integridad referencial:** todos los `customer_id` son válidos.
- **Coherencia temporal:** 16,923 pedidos (50.4%) tenían `order_time` anterior al `signup_date` del cliente; se corrigieron en 3.7 con el mismo criterio que usa Rocío en su módulo.

`orders` no requirió eliminar filas; se aplica `clean_orders` (en `SRC/data_clean.py`) sobre el `orders` ya corregido, ya que revalida todas estas condiciones (imprimiendo el resultado de cada chequeo) y deja el dataset listo para reutilizarse fuera del notebook.

In [15]:
orders = clean_orders(orders, customers)
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 33580 entries, 0 to 33579
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        33580 non-null  int64  
 1   customer_id     33580 non-null  int64  
 2   order_time      33580 non-null  str    
 3   payment_method  33580 non-null  str    
 4   discount_pct    33580 non-null  int64  
 5   subtotal_usd    33580 non-null  float64
 6   total_usd       33580 non-null  float64
 7   country         33580 non-null  str    
 8   device          33580 non-null  str    
 9   source          33580 non-null  str    
dtypes: float64(2), int64(3), str(5)
memory usage: 2.6 MB


In [16]:
products.head()

,product_id,category,name,price_usd,cost_usd,margin_usd
0,1,Electronics,SSD MediumBlue 149,570.28,352.69,217.59
1,2,Electronics,Keyboard DeepPink 696,498.13,263.13,235.00
2,3,Electronics,Headphones Orchid 188,548.53,309.60,238.93
3,4,Electronics,Smartwatch BurlyWood 664,268.36,153.56,114.80
4,5,Electronics,Smartwatch Cornsilk 328,63.69,42.65,21.04


## 4. Limpieza de `products`

### 4.1 Diagnóstico inicial

In [17]:
print('Shape:', products.shape)
print()
print(products.dtypes)
print()
print('Nulos por columna:')
print(products.isnull().sum())
print()
print('Filas duplicadas (exactas):', products.duplicated().sum())
print('product_id duplicados:', products['product_id'].duplicated().sum())

Shape: (1197, 6)

product_id      int64
category          str
name              str
price_usd     float64
cost_usd      float64
margin_usd    float64
dtype: object

Nulos por columna:
product_id    0
category      0
name          0
price_usd     0
cost_usd      0
margin_usd    0
dtype: int64

Filas duplicadas (exactas): 0
product_id duplicados: 0


**Resultado:** sin nulos, sin duplicados, `product_id` único.

### 4.2 Validación de valores numéricos

Se verifica que `price_usd` y `cost_usd` sean positivos y que el costo nunca supere al precio de venta.

In [18]:
print('price_usd <= 0:', (products['price_usd'] <= 0).sum())
print('cost_usd <= 0:', (products['cost_usd'] <= 0).sum())
print('cost_usd > price_usd:', (products['cost_usd'] > products['price_usd']).sum())

price_usd <= 0: 0
cost_usd <= 0: 0
cost_usd > price_usd: 0


**Resultado:** no se encontraron precios ni costos inválidos.

### 4.3 Validación de consistencia (`margin_usd = price_usd − cost_usd`)

In [19]:
calculado = (products['price_usd'] - products['cost_usd']).round(2)
diferencia = (calculado - products['margin_usd']).abs()
print('Filas inconsistentes (> 0.01):', (diferencia > 0.01).sum())

Filas inconsistentes (> 0.01): 0


**Resultado:** `margin_usd` es 100% consistente con `price_usd - cost_usd`.

### 4.4 Validación de texto

Se revisan `category` y `name` en busca de espacios extra o valores fuera de las 7 categorías esperadas.

In [20]:
print('categorias:', sorted(products['category'].unique()))
print('name con espacios al inicio/final:', (products['name'] != products['name'].str.strip()).sum())
print('category con espacios al inicio/final:', (products['category'] != products['category'].str.strip()).sum())

categorias: ['Beauty', 'Books', 'Electronics', 'Fashion', 'Home & Kitchen', 'Sports', 'Toys']
name con espacios al inicio/final: 0
category con espacios al inicio/final: 0


**Resultado:** las 7 categorías son las esperadas y no hay problemas de espacios en `name` ni `category`.

### 4.5 Resumen de la limpieza de `products`

- **Nulos y duplicados:** no había.
- **Valores numéricos:** precios y costos válidos, sin costos mayores al precio.
- **Consistencia de cálculo:** `margin_usd` siempre correcto.
- **Texto:** categorías y nombres limpios.

`products` no requirió eliminar ni modificar filas; se aplica `clean_products` (en `SRC/data_clean.py`) igual, para dejar las mismas validaciones codificadas y reutilizables fuera del notebook.

In [21]:
products = clean_products(products)
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  1197 non-null   int64  
 1   category    1197 non-null   str    
 2   name        1197 non-null   str    
 3   price_usd   1197 non-null   float64
 4   cost_usd    1197 non-null   float64
 5   margin_usd  1197 non-null   float64
dtypes: float64(3), int64(1), str(2)
memory usage: 56.2 KB


In [22]:
customers.head()

,customer_id,name,email,country,age,signup_date,marketing_opt_in
0,1,Jennifer Salinas,nicholas59@example.org,JP,71,2020-09-04,True
1,2,Phillip Ramos,christinarubio@example.com,IN,26,2020-04-05,False
2,3,Dawn Fowler,jessica03@example.org,BR,21,2023-08-31,True
3,4,Mario Butler,paula27@example.org,FR,63,2022-06-30,True
4,5,Amber Brown,kevin85@example.net,BR,19,2022-07-22,True


## 5. Limpieza de `customers`

### 5.1 Diagnóstico inicial

In [23]:
print('Shape:', customers.shape)
print()
print(customers.dtypes)
print()
print('Nulos por columna:')
print(customers.isnull().sum())
print()
print('Filas duplicadas (exactas):', customers.duplicated().sum())
print('customer_id duplicados:', customers['customer_id'].duplicated().sum())
print('email duplicados:', customers['email'].duplicated().sum())

Shape:

 (20000, 7)

customer_id         int64
name                  str
email                 str
country               str
age                 int64
signup_date           str
marketing_opt_in     bool
dtype: object

Nulos por columna:
customer_id         0
name                0
email               0
country             0
age                 0
signup_date         0
marketing_opt_in    0
dtype: int64

Filas duplicadas (exactas): 0
customer_id duplicados: 0
email duplicados: 0


**Resultado:** sin nulos, sin duplicados; `customer_id` y `email` son únicos.

### 5.2 Validación de rango (`age`)

Se revisa que la edad esté en un rango real.

In [24]:
print(customers['age'].describe())
print('age fuera de [18, 100]:', ((customers['age'] < 18) | (customers['age'] > 100)).sum())

count    20000.000000
mean        46.492550
std         16.767961
min         18.000000
25%         32.000000
50%         46.500000
75%         61.000000
max         75.000000
Name: age, dtype: float64


age fuera de [18, 100]: 0


**Resultado:** todas las edades están entre 18 y 75 años.

### 5.3 Validación de formato de `email`

Se valida que todos los emails tengan un formato básico válido.

In [25]:
email_valido = customers['email'].str.match(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
print('emails con formato invalido:', (~email_valido).sum())

emails con formato invalido: 0


**Resultado:** el 100% de los emails tiene formato válido.

### 5.4 Validación de fechas (`signup_date`)

In [26]:
signup_parsed = pd.to_datetime(customers['signup_date'], errors='coerce')
print('Fechas no parseables:', signup_parsed.isnull().sum())
print('Rango de fechas:', signup_parsed.min(), '->', signup_parsed.max())

Fechas no parseables:

 0
Rango de fechas: 2020-01-01 00:00:00 -> 2025-10-31 00:00:00


**Resultado:** todas las fechas de alta son válidas y coherentes con el rango operativo del negocio (2020-2025).

### 5.5 Validación de texto

Se revisan espacios extra en `name`/`email` y que `country` use los mismos códigos que en `orders`.

In [27]:
print('name con espacios al inicio/final:', (customers['name'] != customers['name'].str.strip()).sum())
print('email con espacios al inicio/final:', (customers['email'] != customers['email'].str.strip()).sum())
print('paises:', sorted(customers['country'].unique()))

name con espacios al inicio/final: 0
email con espacios al inicio/final: 0
paises: ['AE', 'AU', 'BR', 'CA', 'DE', 'ES', 'FR', 'GB', 'IN', 'JP', 'MX', 'NL', 'PL', 'SE', 'SG', 'US', 'ZA']


**Resultado:** sin problemas de espacios; los países coinciden con el mismo catálogo de códigos usado en `orders`.

### 5.6 Resumen de la limpieza de `customers`

- **Nulos y duplicados:** no había.
- **Edad:** siempre en rango plausible (18-75).
- **Email:** formato válido en el 100% de los casos.
- **Fechas de alta:** válidas y coherentes.
- **Texto:** sin espacios extra, países consistentes con `orders`.

`customers` no requirió eliminar ni modificar filas; se aplica `clean_customers` (en `SRC/data_clean.py`) igual, para dejar las mismas validaciones codificadas y reutilizables fuera del notebook.

In [28]:
customers = clean_customers(customers)
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   customer_id       20000 non-null  int64
 1   name              20000 non-null  str  
 2   email             20000 non-null  str  
 3   country           20000 non-null  str  
 4   age               20000 non-null  int64
 5   signup_date       20000 non-null  str  
 6   marketing_opt_in  20000 non-null  bool 
dtypes: bool(1), int64(2), str(4)
memory usage: 957.2 KB
